In [ ]:
import asyncio
from pprint import pprint
from tqdm.notebook import tqdm
from typing import Generator, List, Tuple, Union
from itertools import batched
from meipi.indexing import Config, appconf, AsyncFileOperations, DBOperations, DBPool


In [2]:
picroot = "/home/padmin/Bilder/Archiv/11 Bilder - sortiert archiv"
textroot = "/home/padmin/Dokumente"
textpool = DBPool(1,"Texte",textroot,"Pool für Textdateien")
picpool = DBPool(2,"Pics", picroot, "Pool für Bilder")
pprint(appconf)
pool = picpool
relpath = ""
dbop = DBOperations(pool,appconf)

Config(envfile='config.env', pg_host='localhost', pg_port='5432', pg_user='postgres', pg_passwd='postgres', pg_database='postgres', pg_schema='meipi-indexing', pg_api_key='pg-docker', tika_noocr_url='http://localhost:9998', tika_ocrurl='http://localhost:9997', datadir='./data', docsuf={'.htm', '.pdf', '.epub', '.odt', '.docx', '.md', '.txt', '.html', '.doc'}, picsuf={'.png', '.jpeg', '.tif', '.heic', '.tiff', '.jpg', '.bmp'}, vidsuf={'.avi', '.mov', '.mp4', '.mkv', '.vob', '.mcf'}, logger_name='sqlalchemy.engine', loglevel=20, logger=<Logger sqlalchemy.engine (INFO)>)


In [3]:
loop = asyncio.get_event_loop()
loop.set_task_factory(asyncio.eager_task_factory)
async with AsyncFileOperations(pool, appconf) as afop:
    files = afop.dir_tree(relpath)
    batches = batched(files, 500)
    for i, batch in tqdm(enumerate(batches)):
        tasklist = set()
        for file in batch:
            tasklist.add(asyncio.create_task(afop.file_to_db(file)))
        with dbop.Session() as session:
            async for task in asyncio.as_completed(tasklist):
                dbmeta, dbdoc, dbpic, dbvid = await task
                session.add(dbmeta)
                if dbdoc:
                    session.add(dbdoc)
                if dbpic:
                    session.add(dbpic)
            session.flush()
            session.commit()

0it [00:00, ?it/s]

Error <TikaKey.Parsers: 'X-TIKA:Parsed-By'> parsing file /home/padmin/Bilder/Archiv/11 Bilder - sortiert archiv/Bilder 2014/2014-07-Bali - Gesamt/2014-07-22/DSC01174.JPG
Error <TikaKey.Parsers: 'X-TIKA:Parsed-By'> parsing file /home/padmin/Bilder/Archiv/11 Bilder - sortiert archiv/Bilder 2017/2017-06-phi-Sommerfest - Gesamt/P6230058.ORF
Out of bounds IFD
Out of bounds IFD
XML parsing failure
Error  parsing file /home/padmin/Bilder/Archiv/11 Bilder - sortiert archiv/Bilder 1999/1999-07-Seychelles/APS-244-584_00193.tif
Error  parsing file /home/padmin/Bilder/Archiv/11 Bilder - sortiert archiv/Bilder 1999/1999-07-Seychelles/APS-889-189_00985.tif
Error  parsing file /home/padmin/Bilder/Archiv/11 Bilder - sortiert archiv/Bilder 1999/1999-07-Seychelles/APS-244-584_00194.tif
Error  parsing file /home/padmin/Bilder/Archiv/11 Bilder - sortiert archiv/Bilder 1999/1999-07-Seychelles/APS-244-587_00232.tif
Error  parsing file /home/padmin/Bilder/Archiv/11 Bilder - sortiert archiv/Bilder 1999/1999-0